In [1]:
import numpy as np
import glob
import os
import sys
import h5py

In [5]:
def convert_file(filename, L):
    lattice = np.full(L*L, -1, dtype=int)
    with open(filename, 'r') as file:
        curr_id = 0
        # Assign ids to clusters
        for line in file:
            split = line.strip().split()
            if len(split) == 0:
                continue
            curr_posn = int(split[1])
            lattice[curr_posn] = curr_id
            gaps = split[2:]
            for gap in gaps:
                curr_posn += int(gap)
                lattice[curr_posn] = curr_id
            curr_id += 1
        # Give unique ids to empty sites
        mask = (lattice == -1)
        n_empty = mask.sum()
        lattice[mask] = np.arange(curr_id, curr_id + n_empty)
    return lattice

def list_files_scandir(path):
    files = [entry.name for entry in os.scandir(path) if entry.is_file()]
    return files

def convert_run(input_dirname, L):
    samples = list()
    files = list_files_scandir(input_dirname)
    for filename in files:
        in_path = os.path.join(input_dirname, filename)
        samples.append(convert_file(in_path, L))
    return samples

In [8]:
root = "/projects/p32813/blume_capel/data/sweep/t3_d3/spin/96/0"
samples = convert_run(root, 96)
L = 96
with h5py.File("samples.h5", "w") as f:
    dset = f.create_dataset(
        "samples",
        shape=(len(samples), L*L),
        dtype="int8",
        compression="gzip",
        compression_opts=4
    )
    for i in range(len(samples)):
        dset[i] = samples[i]

In [9]:
os.getcwd()


'/gpfs/projects/p32813/blume_capel/notebooks'